# Phase 4 — Model Training & Comparison

**Research question:** *Based on age, stress level, and activity level, can we predict sleep quality?*

**Goal of this notebook:** train the two models the project promised, compare them fairly, and stress-test whether we can trust them. We run two different *paths* to the same three classes (**Low / Medium / High** sleep quality) and see which wins — and whether either beats simply guessing the most common class.

| Path | Model | What it does |
|---|---|---|
| **A** | Linear Regression | predicts the 1–10 *score*, then buckets it into Low / Med / High |
| **B** | Random Forest | classifies Low / Med / High *directly* |

**Why two paths?** Phase 2 found only one linear signal (stress, r ≈ −0.64) and near-zero linear signal from age, steps, and exercise. Linear Regression can only use *linear* relationships — so it's the honest "how far does the obvious signal get us?" baseline. A Random Forest can pick up *non-linear* and *interaction* effects a line can't. Comparing them answers: **is there anything in this data beyond the straight line stress draws?**

**What we check after training** (each is a section below):

1. A head-to-head accuracy comparison against a **majority-class baseline** (always guess Medium) — the bar any model must clear to be useful.
2. **K-fold cross-validation** — is the score stable, or a fluke of one lucky train/test split?
3. **Permutation importance** — which features does each model *actually* rely on? (Phase 2 predicted stress and almost nothing else — this is where we catch a model that disagrees.)
4. A **fairness audit** across occupation and gender — does accuracy hold up for every group, or is the model good on average but bad for some people?

**Reminder:** this dataset is *synthetic*, so everything here describes the patterns built into the data — not proven facts about human sleep.

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    r2_score, root_mean_squared_error, mean_absolute_error,
    accuracy_score, classification_report, confusion_matrix,
)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 117   # same seed the Phase 3 charts used, so results line up

df = pd.read_csv("../sleep_health_dataset.csv")

# The three factors from our research question (activity = steps + whether they exercised)
PREDICTORS = ["age", "stress_score", "steps_that_day", "exercise_day"]
TARGET = "sleep_quality_score"

# Bucket the 1-10 score into three classes, using the SAME cut points as the
# Phase 3 histogram: Low (<=4), Medium (5-6), High (>=7).
BUCKETS = ["Low", "Medium", "High"]
def to_bucket(score):
    return pd.cut(score, bins=[-np.inf, 4.5, 6.5, np.inf], labels=BUCKETS, ordered=True)

df["quality_bucket"] = to_bucket(df[TARGET])

print("class balance (share of all people):")
print(df["quality_bucket"].value_counts(normalize=True).reindex(BUCKETS).round(3))

class balance (share of all people):
quality_bucket
Low       0.415
Medium    0.445
High      0.140
Name: proportion, dtype: float64


## 1. Train / test split

We hold out **20% of the data** as a test set the models never see during training — the only honest way to estimate how they'll do on new people. We **stratify** on the sleep-quality bucket so the train and test sets keep the same class mix — roughly **42 % Low / 44 % Medium / 14 % High** (the mean score is 4.9, so Medium is the most common class and High is rare). Without stratifying, a random split could hand the test set an unrepresentative slice and distort every number below.

We also record the **majority-class baseline**: if you ignored the data entirely and always guessed "Medium" (the most common class), how often would you be right? Every model has to beat this to be worth anything.

In [2]:
X = df[PREDICTORS]
y_score = df[TARGET]          # for Path A (regression)
y_bucket = df["quality_bucket"]  # for Path B (classification)

X_train, X_test, ys_train, ys_test, yb_train, yb_test = train_test_split(
    X, y_score, y_bucket,
    test_size=0.20, random_state=RANDOM_STATE, stratify=y_bucket,
)

baseline = yb_test.value_counts(normalize=True).max()
print(f"train: {len(X_train):,} rows    test: {len(X_test):,} rows")
print(f"majority-class baseline (always guess 'Medium'): {baseline:.1%}")

train: 80,000 rows    test: 20,000 rows
majority-class baseline (always guess 'Medium'): 44.5%


## 2. Path A — Linear Regression

Linear Regression predicts the sleep-quality *score* as a straight-line combination of the four features. We read it in two stages:

**As a regression** — three standard numbers:
- **R²** — the share of the variation in sleep quality the model explains (1.0 = perfect, 0 = no better than always predicting the mean).
- **RMSE** — typical error size, in points on the 1–10 scale.
- **MAE** — average absolute error, also in points.

The **coefficients** tell the story directly: each is how much the predicted score moves per one-unit rise in that feature. If Phase 2 was right, `stress_score` should carry a clear negative coefficient and the rest should sit near zero.

In [3]:
linreg = LinearRegression().fit(X_train, ys_train)
pred_score = linreg.predict(X_test)

print("Regression fit on the 1-10 score:")
print(f"  R2   = {r2_score(ys_test, pred_score):.3f}")
print(f"  RMSE = {root_mean_squared_error(ys_test, pred_score):.3f}  points (1-10 scale)")
print(f"  MAE  = {mean_absolute_error(ys_test, pred_score):.3f}  points")

coefs = pd.Series(linreg.coef_, index=PREDICTORS).sort_values()
print("\nCoefficients (change in predicted score per 1-unit increase in the feature):")
print(coefs.round(4))
print(f"intercept = {linreg.intercept_:.3f}")

Regression fit on the 1-10 score:
  R2   = 0.415
  RMSE = 1.153  points (1-10 scale)
  MAE  = 0.931  points

Coefficients (change in predicted score per 1-unit increase in the feature):
stress_score     -0.5939
age              -0.0036
steps_that_day    0.0000
exercise_day      0.0670
dtype: float64
intercept = 8.370


Now the second stage: **bucket the predicted scores** with the same cut points, so Path A produces Low / Med / High labels we can compare head-to-head with the Random Forest. The `classification_report` breaks accuracy down per class:

- **precision** — of the people the model *called* High, how many really were?
- **recall** — of the people who *were* High, how many did the model catch?
- **f1** — the balance of the two; **support** — how many test people are in that class.

In [4]:
pred_bucket_A = to_bucket(pd.Series(pred_score, index=ys_test.index))
acc_A = accuracy_score(yb_test, pred_bucket_A)

print(f"Path A accuracy (regression -> buckets): {acc_A:.1%}\n")
print(classification_report(yb_test, pred_bucket_A, labels=BUCKETS, zero_division=0))

Path A accuracy (regression -> buckets): 61.4%



              precision    recall  f1-score   support

         Low       0.70      0.60      0.65      8304
      Medium       0.56      0.73      0.63      8899
        High       0.65      0.28      0.39      2797

    accuracy                           0.61     20000
   macro avg       0.64      0.54      0.56     20000
weighted avg       0.63      0.61      0.60     20000



## 3. Path B — Random Forest

A Random Forest is an ensemble of decision trees, each trained on a random slice of the data and features; the class most trees vote for wins. Unlike the straight line in Path A, it can bend to **non-linear** patterns and **interactions** between features. It's trained directly on the Low / Med / High labels — no scores, no bucketing afterward.

If the forest does much better than Path A, that's evidence of real non-linear structure. If it does about the same, it confirms Phase 2's verdict: the signal here is essentially the single straight line stress draws.

In [5]:
rf = RandomForestClassifier(
    n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1,
)
rf.fit(X_train, yb_train)
pred_bucket_B = rf.predict(X_test)

acc_B = accuracy_score(yb_test, pred_bucket_B)
print(f"Path B accuracy (Random Forest classifier): {acc_B:.1%}\n")
print(classification_report(yb_test, pred_bucket_B, labels=BUCKETS, zero_division=0))

Path B accuracy (Random Forest classifier): 55.0%



              precision    recall  f1-score   support

         Low       0.61      0.61      0.61      8304
      Medium       0.53      0.55      0.54      8899
        High       0.43      0.36      0.39      2797

    accuracy                           0.55     20000
   macro avg       0.52      0.51      0.51     20000
weighted avg       0.55      0.55      0.55     20000



### Confusion matrices

A confusion matrix shows *where* each model's mistakes land: rows are the true class, columns are what the model predicted. The diagonal is correct predictions; everything off-diagonal is an error. We especially want to see that mistakes are "near misses" (Medium↔High) rather than the model confusing Low with High.

In [6]:
for name, preds in [("Path A - Linear Regression", pred_bucket_A),
                    ("Path B - Random Forest", pred_bucket_B)]:
    cm = confusion_matrix(yb_test, preds, labels=BUCKETS)
    table = pd.DataFrame(cm,
                         index=[f"true {b}" for b in BUCKETS],
                         columns=[f"pred {b}" for b in BUCKETS])
    print(name)
    display(table)

Path A - Linear Regression


,pred Low,pred Medium,pred High
true Low,5014,3271,19
true Medium,2013,6490,396
true High,116,1905,776


Path B - Random Forest


,pred Low,pred Medium,pred High
true Low,5090,2912,302
true Medium,2954,4884,1061
true High,332,1448,1017


## 4. Head-to-head — do the models beat the baseline?

The single most important comparison in the whole project. A model that can't clear the "always guess Medium" bar has learned nothing useful. **Lift** is how many percentage points of accuracy each path adds on top of that baseline.

In [7]:
comparison = pd.DataFrame(
    {"accuracy": [baseline, acc_A, acc_B]},
    index=["Baseline (always Medium)", "Path A - LinReg -> buckets", "Path B - Random Forest"],
)
comparison["lift over baseline"] = comparison["accuracy"] - baseline
display(comparison.round(3))

,accuracy,lift over baseline
Baseline (always Medium),0.445,0.000
Path A - LinReg -> buckets,0.614,0.169
Path B - Random Forest,0.550,0.105


## 5. K-fold cross-validation — is the score stable?

A single train/test split could be lucky or unlucky. **5-fold cross-validation** cuts the data into 5 parts, trains on 4 and tests on the 5th, and rotates so every row is tested exactly once. We report the mean **and** the spread (±) across folds:

- We score **Path A by R²** (it's a regression) and **Path B by accuracy** (it's a classifier) — each on the metric that fits it.
- A **small spread** means the result is stable and trustworthy; a large one would mean the model's performance depends heavily on which rows it happened to see.

(The classifier uses *stratified* folds so each fold keeps the class balance.)

In [8]:
# Path A: regression R2 across 5 plain folds
r2_scores = cross_val_score(LinearRegression(), X, y_score, cv=5, scoring="r2")

# Path B: classifier accuracy across 5 stratified folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
acc_scores = cross_val_score(
    RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    X, y_bucket, cv=skf, scoring="accuracy",
)

print(f"Path A  regression R2  per fold: {np.round(r2_scores, 3)}")
print(f"                        mean {r2_scores.mean():.3f}  (+/- {r2_scores.std():.3f})\n")
print(f"Path B  RF accuracy    per fold: {np.round(acc_scores, 3)}")
print(f"                        mean {acc_scores.mean():.3f}  (+/- {acc_scores.std():.3f})")

Path A  regression R2  per fold: [0.407 0.411 0.41  0.407 0.411]
                        mean 0.409  (+/- 0.002)

Path B  RF accuracy    per fold: [0.55  0.542 0.55  0.546 0.556]
                        mean 0.549  (+/- 0.005)


## 6. Permutation importance — what does each model actually use?

Feature importance done the honest way: shuffle one feature's values (breaking its link to the target) and measure how much the model's score *drops*. A big drop means the model leaned on that feature; ~0 means it was ignoring it. We measure the drop in the metric that fits each path — **R² for Path A, accuracy for Path B**.

This is the check Phase 2 set up for us: the stats said **stress and essentially nothing else**. If either model shows age or steps as important here, that contradicts the correlation table (and the flat scatter plots in Phase 3) — a red flag worth investigating.

In [9]:
perm_A = permutation_importance(linreg, X_test, ys_test, scoring="r2",
                                n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_B = permutation_importance(rf, X_test, yb_test, scoring="accuracy",
                                n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)

importance = pd.DataFrame({
    "Path A (drop in R2)": perm_A.importances_mean,
    "Path B (drop in accuracy)": perm_B.importances_mean,
}, index=PREDICTORS).sort_values("Path B (drop in accuracy)", ascending=False)

display(importance.round(4))

,Path A (drop in R2),Path B (drop in accuracy)
stress_score,0.8248,0.1517
age,0.0010,0.0022
exercise_day,0.0007,0.0016
steps_that_day,0.0000,-0.0007


## 7. Fairness audit — does accuracy hold for everyone?

A model can look good *on average* and still fail specific groups. Before anyone acts on a model, we check that its accuracy holds across the groups in the data. We recompute each path's accuracy **within** each gender and each occupation, on the held-out test set. What we're looking for:

- A large **spread** between the best- and worst-served group is a fairness problem, even if overall accuracy is fine.
- Very small groups (low `n`) will have noisier numbers — we show `n` so we don't over-read them.

In [10]:
audit = pd.DataFrame({
    "true":   yb_test.values,
    "pred_A": np.asarray(pred_bucket_A),
    "pred_B": np.asarray(pred_bucket_B),
    "gender":     df.loc[X_test.index, "gender"].values,
    "occupation": df.loc[X_test.index, "occupation"].values,
})

def group_accuracy(frame, group_col):
    rows = []
    for g, sub in frame.groupby(group_col, observed=True):
        rows.append({
            group_col: g, "n": len(sub),
            "acc_A": accuracy_score(sub["true"], sub["pred_A"]),
            "acc_B": accuracy_score(sub["true"], sub["pred_B"]),
        })
    return (pd.DataFrame(rows)
            .sort_values("acc_B", ascending=False)
            .reset_index(drop=True))

print("Accuracy by gender:")
display(group_accuracy(audit, "gender").round(3))

Accuracy by gender:


,gender,n,acc_A,acc_B
0,Female,9996,0.614,0.555
1,Male,9604,0.614,0.545
2,Other,400,0.605,0.525


In [11]:
occ = group_accuracy(audit, "occupation").round(3)
print("Accuracy by occupation:")
display(occ)
print(f"Path A spread across occupations: {occ['acc_A'].max() - occ['acc_A'].min():.1%}")
print(f"Path B spread across occupations: {occ['acc_B'].max() - occ['acc_B'].min():.1%}")

Accuracy by occupation:


,occupation,n,acc_A,acc_B
0,Lawyer,1003,0.711,0.653
1,Nurse,2003,0.694,0.646
2,Doctor,1563,0.681,0.616
3,Retired,1438,0.614,0.583
4,Driver,1397,0.637,0.563
5,Student,2971,0.601,0.535
6,Manager,1587,0.580,0.520
7,Sales,1380,0.591,0.516
8,Homemaker,1199,0.590,0.515
9,Software Engineer,2443,0.571,0.502


Path A spread across occupations: 15.0%
Path B spread across occupations: 16.5%


## 8. Findings

*(These conclusions match the executed output above; re-run the notebook if you change anything upstream.)*

- **Both models clear the ~44.5% majority-class baseline** (always guessing Medium), so each has learned something real — but modestly, exactly what Phase 2 predicted from a dataset with one strong signal and three flat ones.
- **The simpler model wins.** Path A (Linear Regression → buckets) hits **~61%** vs the Random Forest's **~55%**. That looks backwards — the flexible model lost — but it makes sense: sleep quality is **ordinal** (Low < Medium < High), so predicting the continuous score and *then* bucketing respects that ordering and leans on the single clean linear signal (stress). The forest treats the three classes as unordered and spends its extra flexibility fitting noise in the flat features (age, steps), which generalizes *worse* — it especially struggles with the rare High class (recall ~0.36). **Take-away for the slide: when the signal is essentially one straight line, a straight line is the right tool.**
- **Cross-validation is tight** (Path A R² ±0.002, Path B accuracy ±0.005 across folds), so these scores are stable properties of the data, not artifacts of one lucky split.
- **Permutation importance confirms the Phase 2 story:** `stress_score` dominates both models; `age`, `steps_that_day`, and `exercise_day` sit at ≈ 0. Neither model contradicts the statistics — the guardrail we built in Phase 2 holds.
- **Fairness — the one thing to watch:** accuracy is essentially equal across **gender** (~1 pt spread). But it varies **~16 points across occupation** — both models predict best for Lawyers/Nurses/Doctors (~0.65–0.71) and worst for Teachers/Freelancers/Software Engineers (~0.49–0.56). Occupation isn't even a feature, so this reflects how well *stress alone* explains each group's sleep — it's more predictable for some occupations than others. Worth naming honestly rather than reporting only the flattering average.

**Answer to the research question:** *partly.* We can predict sleep-quality class better than chance, but the predictive power comes **almost entirely from stress** — age and activity add essentially nothing in this dataset. A simple, interpretable model is enough; the Random Forest's extra flexibility actually hurt, which is itself a finding worth a slide.

**Honest caveats:** the data is synthetic, so this really describes the generator's built-in patterns; "activity" was captured only as steps + an exercise flag, so a richer measure might behave differently; and the per-occupation accuracy gap deserves a closer look before anyone acts on a model like this.